# Foundations Summary

Synthesize verified findings from the foundations labs without introducing unmeasured claims.

## Objectives

Connect CPU, memory, CUDA, profiling, and RDMA observations and identify the next experiments they justify.

## Background

System performance emerges from interactions among compute, memory, storage, scheduling, accelerator execution, and communication.

## Prediction

The completed foundations labs will not support a single universal performance bottleneck. Instead, the dominant constraint should depend on workload scale and execution phase:

- small CPU workloads will be sensitive to core selection, affinity, interpreter overhead, and cache locality;
- sufficiently large CPU and GPU workloads will increasingly expose memory bandwidth and access-pattern effects;
- small CUDA operations will be disproportionately affected by launch and synchronization overhead;
- independent GPU work may overlap only when resource use and dependencies leave concurrency available;
- profiling should show that host-side, device-side, and synchronization time must be measured separately;
- communication between two DGX Spark systems will remain much slower than local memory movement, making communication volume and synchronization frequency important for distributed LLM workloads.

Before treating any of these expectations as findings, this notebook will verify which preceding notebooks contain executed measurements and completed interpretations on the current `main` branch.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [ ]:
import json
from pathlib import Path

import pandas as pd


foundations_directory = repository_root / "experiments" / "00-foundations"

source_notebooks = sorted(
    path
    for path in foundations_directory.glob("[0-9][0-9]-*.ipynb")
    if path.name != "10-summary.ipynb"
)

expected_notebooks = [f"{index:02d}" for index in range(10)]

discovered_prefixes = [path.name.split("-", maxsplit=1)[0] for path in source_notebooks]

if discovered_prefixes != expected_notebooks:
    raise RuntimeError(
        f"Expected notebooks 00 through 09, but found {discovered_prefixes}"
    )


def markdown_text(cell: dict[str, object]) -> str:
    return "".join(cell.get("source", [])).strip()


def code_text(cell: dict[str, object]) -> str:
    return "".join(cell.get("source", []))


def output_contains_error(cell: dict[str, object]) -> bool:
    return any(
        output.get("output_type") == "error" for output in cell.get("outputs", [])
    )


def section_body(
    cells: list[dict[str, object]],
    heading: str,
) -> str | None:
    section_lines: list[str] = []
    collecting = False

    for cell in cells:
        if cell.get("cell_type") != "markdown":
            continue

        text = markdown_text(cell)
        lines = text.splitlines()

        if not lines:
            continue

        first_line = lines[0].strip()

        if collecting and first_line.startswith("## "):
            break

        if first_line == heading:
            collecting = True
            section_lines.extend(lines[1:])
        elif collecting:
            section_lines.extend(lines)

    body = "\n".join(section_lines).strip()
    return body or None


inventory_rows: list[dict[str, object]] = []
section_rows: list[dict[str, object]] = []

section_headings = [
    "## Prediction",
    "## Observations",
    "## Explanation",
    "## Connection to LLMs",
    "## Further Exploration",
]

for notebook_path in source_notebooks:
    notebook = json.loads(notebook_path.read_text())
    cells = notebook["cells"]

    code_cells = [cell for cell in cells if cell.get("cell_type") == "code"]
    markdown_cells = [cell for cell in cells if cell.get("cell_type") == "markdown"]

    executed_code_cells = [
        cell for cell in code_cells if cell.get("execution_count") is not None
    ]
    error_cells = [cell for cell in code_cells if output_contains_error(cell)]

    all_source = "\n".join(
        markdown_text(cell) if cell.get("cell_type") == "markdown" else code_text(cell)
        for cell in cells
    )

    sections = {
        heading.removeprefix("## "): section_body(cells, heading)
        for heading in section_headings
    }

    inventory_rows.append(
        {
            "notebook": notebook_path.name,
            "markdown_cells": len(markdown_cells),
            "code_cells": len(code_cells),
            "executed_code_cells": len(executed_code_cells),
            "execution_coverage": (
                len(executed_code_cells) / len(code_cells) if code_cells else None
            ),
            "error_cells": len(error_cells),
            "todo_occurrences": all_source.count("TODO"),
            "has_observations": bool(sections["Observations"]),
            "observations_are_todo": (
                sections["Observations"] is not None
                and "TODO" in sections["Observations"]
            ),
            "has_explanation": bool(sections["Explanation"]),
            "explanation_is_todo": (
                sections["Explanation"] is not None
                and "TODO" in sections["Explanation"]
            ),
        }
    )

    for section_name, body in sections.items():
        section_rows.append(
            {
                "notebook": notebook_path.name,
                "section": section_name,
                "present": body is not None,
                "contains_todo": body is not None and "TODO" in body,
                "character_count": len(body) if body is not None else 0,
                "body": body,
            }
        )


notebook_inventory = pd.DataFrame(inventory_rows)
recorded_sections = pd.DataFrame(section_rows)

notebook_inventory

,notebook,markdown_cells,code_cells,executed_code_cells,execution_coverage,error_cells,todo_occurrences,has_observations,observations_are_todo,has_explanation,explanation_is_todo
0,00-machine-overview.ipynb,10,2,2,1.000000,0,4,True,True,True,True
1,01-cpu-architecture.ipynb,10,2,2,1.000000,0,5,True,True,True,True
2,02-memory-hierarchy.ipynb,10,28,27,0.964286,0,1,True,False,True,False
3,03-linux-memory.ipynb,10,28,18,0.642857,0,1,True,False,True,False
4,04-vectorization.ipynb,10,17,11,0.647059,0,1,True,False,True,False
5,05-cuda-fundamentals.ipynb,18,22,10,0.454545,0,0,True,False,True,False
6,06-gpu-memory.ipynb,15,18,6,0.333333,0,1,True,False,True,False
7,07-cuda-streams.ipynb,14,19,9,0.473684,0,0,True,False,True,False
8,08-profiling.ipynb,27,40,29,0.725000,0,0,True,False,True,False
9,09-rdma-fundamentals.ipynb,25,69,46,0.666667,0,0,True,False,True,False


In [ ]:
inventory_summary = pd.Series(
    {
        "source_notebooks": len(notebook_inventory),
        "fully_executed_notebooks": int(
            (notebook_inventory["execution_coverage"] == 1.0).sum()
        ),
        "notebooks_with_errors": int((notebook_inventory["error_cells"] > 0).sum()),
        "notebooks_with_todos": int((notebook_inventory["todo_occurrences"] > 0).sum()),
        "completed_observation_sections": int(
            (
                notebook_inventory["has_observations"]
                & ~notebook_inventory["observations_are_todo"]
            ).sum()
        ),
        "completed_explanation_sections": int(
            (
                notebook_inventory["has_explanation"]
                & ~notebook_inventory["explanation_is_todo"]
            ).sum()
        ),
    },
    name="count",
)

inventory_summary.to_frame()

,count
source_notebooks,10
fully_executed_notebooks,2
notebooks_with_errors,0
notebooks_with_todos,6
completed_observation_sections,8
completed_explanation_sections,8


In [ ]:
narrative_status = (
    recorded_sections.pivot(
        index="notebook",
        columns="section",
        values="contains_todo",
    )
    .rename_axis(columns=None)
    .reset_index()
)

for column in section_headings:
    section_name = column.removeprefix("## ")
    if section_name in narrative_status:
        narrative_status[section_name] = narrative_status[section_name].map(
            {
                False: "recorded",
                True: "TODO",
            }
        )

narrative_status

,notebook,Connection to LLMs,Explanation,Further Exploration,Observations,Prediction
0,00-machine-overview.ipynb,recorded,TODO,TODO,TODO,TODO
1,01-cpu-architecture.ipynb,recorded,TODO,TODO,TODO,TODO
2,02-memory-hierarchy.ipynb,recorded,recorded,TODO,recorded,recorded
3,03-linux-memory.ipynb,recorded,recorded,TODO,recorded,recorded
4,04-vectorization.ipynb,recorded,recorded,TODO,recorded,recorded
5,05-cuda-fundamentals.ipynb,recorded,recorded,recorded,recorded,recorded
6,06-gpu-memory.ipynb,recorded,recorded,TODO,recorded,recorded
7,07-cuda-streams.ipynb,recorded,recorded,recorded,recorded,recorded
8,08-profiling.ipynb,recorded,recorded,recorded,recorded,recorded
9,09-rdma-fundamentals.ipynb,recorded,recorded,recorded,recorded,recorded


### Interpretation criterion

An executed code cell is evidence that code ran, but it is not automatically evidence for a conclusion. Likewise, a completed prose section may contain either a measured observation or an architectural interpretation.

The inventory therefore answers only whether usable material appears to exist. Each candidate finding must subsequently be checked against its originating cell outputs. Findings will be classified as:

1. **Measured** — directly visible in an output from the recorded run.
2. **Derived** — calculated from recorded measurements using an explicit transformation.
3. **Architectural inference** — a plausible explanation supported by system structure but not directly measured.
4. **Unresolved** — suggested by the experiment but not established by its outputs.

## Observations

TODO: Summarize measured findings and link each statement to its originating run.

## Explanation

TODO: Build a cross-layer explanation and distinguish evidence from remaining hypotheses.

## Connection to LLMs

Use the verified constraints to motivate concrete experiments in distributed inference, training, and communication.

## Further Exploration

TODO: Rank follow-up experiments by expected learning value, cost, and dependency on unresolved questions.